# k-Nearest Neighbours, Support Vector Machines and Naive Bayes – Titanic

**Dataset:** Titanic – 891 passengers (Kaggle), a classic classification example. File: `../00_Data/titanic.csv`.

**Question:** Can the survival of a Titanic passenger be predicted from a few characteristics with distance-based
methods (k-NN, SVM) and with Naive Bayes?

**After this exercise you can ...**
- apply k-NN, explain why features must be scaled and choose $k$ by cross-validation,
- explain the role of the margin, the support vectors, $C$ and the kernel of an SVM,
- compute a Naive Bayes classification by hand and apply Naive Bayes to categorical data.

## Concept

The concepts behind this exercise (formulas and worked examples) are
explained in the notebook [knn_naive_bayes_svm_concept.ipynb](knn_naive_bayes_svm_concept.ipynb). Read it before you start.

## Libraries and settings

In [ ]:
# Libraries
import os
import platform
import warnings
from datetime import datetime
from platform import python_version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import CategoricalNB
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
)
from sklearn.metrics import confusion_matrix
from sklearn.inspection import DecisionBoundaryDisplay

# Ignore warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: k-NN and Naive Bayes by hand

*Time: about 15 minutes. Use the notebook cell below as your calculator: type the formulas with the numbers in
Python (e.g. `(3 + 5) / 2` or `0.4 ** 2`) – no paper needed.*

**a) k-NN with and without scaling.** A new passenger is 30 years old and paid a fare of 40 pounds. Two known
passengers:

| passenger | survived? | age [years] | fare [£] |
|---|---|---|---|
| P1 | no | 58 | 41 |
| P2 | yes | 29 | 70 |

1. Compute the Euclidean distances $\sqrt{\Delta\text{age}^2 + \Delta\text{fare}^2}$ of the new passenger to P1 and P2.
   Who is the nearest neighbour ($k = 1$)?
2. Standardise the differences: divide $\Delta$age by 13 and $\Delta$fare by 50 (the standard deviations) and compute
   the distances again. Who is the nearest neighbour now? Why does the result change?

**b) Naive Bayes.** Of the 891 passengers, 342 survived and 549 died. Among the survivors there were 233 women and 136
first-class passengers; among those who died there were 81 women and 80 first-class passengers. Compute the Naive Bayes
scores for a **woman in 1st class**:

score(survived) = P(survived) · P(woman | survived) · P(1st class | survived), score(died) analogously.

Her probability of survival is score(survived) / (score(survived) + score(died)). Compare it with the observed survival
rate of women in 1st class (91 of 94).

In [ ]:
# a) Distances of the new passenger (age 30, fare 40) to
#    P1 (age 58, fare 41) and P2 (age 29, fare 70) on the original scale
# Your code here

# Distances with standardised differences (age / 13, fare / 50)
# Your code here

# b) Naive Bayes scores and probability of survival of a woman in 1st class
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 2: Prepare the Titanic data

a) Import the Titanic data and prepare the features `male`, `Age` (missing values replaced by the median), `Pclass`,
`SibSp`, `Parch` and `Fare`. Split the data into a stratified training (80 %) and test set (20 %) with `random_state=42`.

b) Create a scatter plot of `Age` against `Fare` (logarithmic y-axis), coloured by `Survived`, and compare the scales of
the features.

In [ ]:
# a) Import the data as titanic and prepare the features
#    File: '../00_Data/titanic.csv'
#    Create the column male, fill the missing ages with the median and
#    split the data (stratified).
features = ['male', 'Age', 'Pclass', 'SibSp', 'Parch', 'Fare']
# Your code here
X_train, X_test, y_train, y_test = ___

# b) Scatter plot Age vs. Fare (log scale; +1 because some fares are 0)
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 3: k-NN – the effect of scaling and the choice of k

a) Compute the 5-fold CV accuracy (on the training set) of `KNeighborsClassifier(n_neighbors=5)` on the **unscaled**
features and of `make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))`. Explain the difference.

b) Compute the 5-fold CV accuracy of the scaled k-NN for $k = 1, 2, \dots, 40$ and plot it against $k$. Choose $k$ and
compute accuracy and confusion matrix on the test set.

c) Predict the survival probability of a 30-year-old man in 3rd class without relatives (fare 8 £) and of a
30-year-old woman in 1st class without relatives (fare 80 £).

In [ ]:
# Exercise 3: 5-fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# a) k-NN without and with scaling (the scaler is part of the pipeline)
# Your code here

# b) CV accuracy of the scaled k-NN for k = 1, ..., 40; best k on the test set
#    (store the final model as knn)
ks = range(1, 41)
cv_acc = []
# Your code here

# c) Two new passengers: man in 3rd class, woman in 1st class
new_passengers = pd.DataFrame(
    {
        'male': [1, 0],
        'Age': [30, 30],
        'Pclass': [3, 1],
        'SibSp': [0, 0],
        'Parch': [0, 0],
        'Fare': [8, 80],
    }
)
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 4: Support vector machines – C and kernel

a) Compute the 5-fold CV accuracy and the number of support vectors of
`make_pipeline(StandardScaler(), SVC(kernel=..., C=...))` for `kernel` ∈ {`'linear'`, `'rbf'`} and
$C$ ∈ {0.01, 1, 100} on the training set (all six features).

b) To *see* the boundaries, fit the same six models on only the two features `Age` and `log_fare = log(1 + Fare)` and
plot the decision regions in a 2 × 3 grid (the code for the grid is given), with the support vectors circled.

c) Describe the influence of $C$ and of the kernel. Which model would you choose?

In [ ]:
# a) CV accuracy and number of support vectors for each kernel and C
# Your code here

# b) Only two features: Age and log(1 + Fare)
X2_train = pd.DataFrame(
    {'Age': X_train['Age'], 'log_fare': np.log1p(X_train['Fare'])}
)

# Figure showing the decision regions for each kernel and C (2 x 3 plots)
plt.figure(figsize=(16, 9))
i = 1
for kernel in ['linear', 'rbf']:
    for C in [0.01, 1, 100]:
        svm = ___
        ax = plt.subplot(2, 3, i)
        DecisionBoundaryDisplay.from_estimator(svm, X2_train, alpha=0.3, ax=ax)

        # Plot the training points (coloured by Survived) and circle the
        # support vectors (svm[-1].support_)
        # Your code here
        i = i + 1
plt.show()

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 5: Naive Bayes with categorical features

a) Fit a `CategoricalNB(alpha=1e-10)` (practically no smoothing) on the two features `male` and `Pclass` of **all**
passengers and predict the survival probability of a woman in 1st class. Compare with your hand calculation of
Exercise 1b.

b) Compute the 5-fold CV accuracy of a `CategoricalNB()` with the features `male`, `Pclass` and `age_group`
(0 = child under 10, 1 = 10–59 years, 2 = 60 years and older) on the training set. Compare with k-NN and SVM.

In [ ]:
# a) CategoricalNB on male and Pclass (all passengers)
X_cat = pd.DataFrame(
    {'male': titanic['male'], 'Pclass': titanic['Pclass'] - 1}
)  # categories must start at 0
# Your code here

# b) CategoricalNB with the features male, Pclass and age_group
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 6: Reflection

Compare k-NN, SVM and Naive Bayes: for which of the three methods is

a) feature scaling necessary,

b) the training fast but the prediction slow,

c) a very large number of features (e.g. thousands of words) unproblematic?

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')